In [3]:
"""
Preprocessing Pipeline — Monetary Policy Transmission to Indian Yield Curve
============================================================================
Inputs  : Raw CSV files for 1Y, 3Y, 5Y, 10Y G-sec yields + Repo rate
Outputs :
  - data/monthly_panel.csv        → main dataset for DNS + SVAR (228 obs)
  - data/daily_panel.csv          → event-study dataset (weekdays only)
  - data/summary_stats.csv        → descriptive statistics table (paper-ready)
  - data/stationarity_report.csv  → ADF + PP unit root results (paper-ready)
  - plots/yield_series.png        → time-series plot of all yields + repo
  - plots/yield_changes.png       → first-difference series
  - plots/correlation_heatmap.png → correlation matrices (levels + diffs)

Dependencies: pandas, numpy, scipy, matplotlib, seaborn
"""

import os
import shutil
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

# ── 0. CONFIG ────────────────────────────────────────────────────────────────

# Once you have all four yield files, add them here
RAW_FILES = {
    "Y10": "/Users/vanshaj/Work/Eco Paper/Preprocessed/India 10-Year Bond Yield Historical Data (1)_standardized_date.csv",
    "Y1" : "/Users/vanshaj/Work/Eco Paper/Preprocessed/India 1-Year Bond Yield Historical Data (1)_standardized_date.csv",
    "Y3" : "/Users/vanshaj/Work/Eco Paper/Preprocessed/India 3-Year Bond Yield Historical Data (1)_standardized_date.csv",
    "Y5" : "/Users/vanshaj/Work/Eco Paper/Preprocessed/India 5-Year Bond Yield Historical Data (1)_standardized_date.csv",
}
REPO_FILE   = "data/raw/Repo_standardized.csv"
STUDY_START = "2006-04-01"
STUDY_END   = "2025-03-31"

MATURITY_LABELS = {
    "Y1"  : "1-Year G-Sec",
    "Y3"  : "3-Year G-Sec",
    "Y5"  : "5-Year G-Sec",
    "Y10" : "10-Year G-Sec",
    "Repo": "RBI Repo Rate",
}
COLORS = {
    "Y1"  : "#2e8b57",
    "Y3"  : "#d4813a",
    "Y5"  : "#8b3a8b",
    "Y10" : "#1f5fa6",
    "Repo": "#c0392b",
}

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data",     exist_ok=True)
os.makedirs("plots",    exist_ok=True)

# ── 1. STAGE RAW FILES ───────────────────────────────────────────────────────
UPLOAD_DIR = "/mnt/user-data/uploads"
UPLOAD_MAP = {
    "Y10": "India_10-Year_Bond_Yield_Historical_Data__1__standardized_date.csv",
    "Y1" : "India_1-Year_Bond_Yield_Historical_Data__1__standardized_date.csv",
    "Y3" : "India_3-Year_Bond_Yield_Historical_Data__1__standardized_date.csv",
    "Y5" : "India_5-Year_Bond_Yield_Historical_Data__1__standardized_date.csv",
    "Repo": "Repo_standardized.csv",
}

def _stage_upload_file(label: str, filename: str) -> str:
    src = os.path.join(UPLOAD_DIR, filename)
    dst = os.path.join("data/raw", filename)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"  Staged {filename} to {dst}")
        return dst
    if os.path.exists(dst):
        return dst
    warnings.warn(
        f"Upload file not found: {src}. "
        f"Will use existing configured path for {label} if available."
    )
    return dst


for label, filename in UPLOAD_MAP.items():
    staged_path = _stage_upload_file(label, filename)
    if label == "Repo":
        REPO_FILE = staged_path
    elif label in RAW_FILES:
        RAW_FILES[label] = staged_path

# ── 2. LOADERS ───────────────────────────────────────────────────────────────

def load_gsec(path: str, label: str) -> pd.Series:
    """
    Load a G-sec yield CSV.
    - Keeps only 'Price' column (yield in %)
    - Drops weekends (Sat/Sun carry Friday values forward — not real obs)
    - Drops Open/High/Low (always equal to Price for G-sec daily data)
    - Drops Change % (string column; recomputed from Price when needed)
    - Restricts to STUDY_START – STUDY_END
    Returns a named pd.Series indexed by Date.
    """
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])

    # Drop weekends (Mon=0 … Fri=4, Sat=5, Sun=6)
    df = df[df["Date"].dt.dayofweek < 5].copy()

    # Restrict to study window
    df = df[(df["Date"] >= STUDY_START) & (df["Date"] <= STUDY_END)].copy()

    # Keep only Date + Price, rename to maturity label
    df = (df[["Date", "Price"]]
            .rename(columns={"Price": label})
            .set_index("Date")
            .sort_index())

    assert not df.index.duplicated().any(), f"Duplicate dates found in {label}"
    print(f"  {label:5s}  rows={len(df):,}  "
          f"NaN={df[label].isna().sum()}  "
          f"range=[{df[label].min():.3f}, {df[label].max():.3f}]%")
    return df[label]


def load_repo(path: str) -> pd.Series:
    """
    Load RBI Repo rate CSV.
    Weekend rows carry Friday's rate forward — drop them.
    """
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df[df["Date"].dt.dayofweek < 5].copy()
    df = df[(df["Date"] >= STUDY_START) & (df["Date"] <= STUDY_END)].copy()
    df = (df[["Date", "Repo"]]
            .set_index("Date")
            .sort_index())
    assert not df.index.duplicated().any(), "Duplicate dates found in Repo"
    print(f"  {'Repo':5s}  rows={len(df):,}  "
          f"NaN={df['Repo'].isna().sum()}  "
series_list = [load_gsec(path, label) for label, path in RAW_FILES.items()]

if os.path.exists(REPO_FILE):
    series_list.append(load_repo(REPO_FILE))
else:
    warnings.warn(
        f"Repo file not found at {REPO_FILE}. "
        "Proceeding without Repo rate. Add the file to data/raw or uploads to include it."
    )


# ── 3. BUILD DAILY PANEL ─────────────────────────────────────────────────────

print("\n[1/5] Loading raw files ...")
series_list = [load_gsec(path, label) for label, path in RAW_FILES.items()]
series_list.append(load_repo(REPO_FILE))

daily = pd.concat(series_list, axis=1).sort_index()

# Indian trading holidays fall on weekdays → appear as NaN after alignment
n_before    = len(daily)
daily_clean = daily.dropna()
n_dropped   = n_before - len(daily_clean)
print(f"\n  Rows before dropna : {n_before:,}")
print(f"  Rows dropped (hols): {n_dropped:,}")
print(f"  Final daily rows   : {len(daily_clean):,}")

daily_clean.to_csv("data/daily_panel.csv")
print("  Saved  data/daily_panel.csv")


# ── 4. BUILD MONTHLY PANEL ───────────────────────────────────────────────────
# Month-end last observation — standard convention in term structure literature

print("\n[2/5] Building monthly panel ...")
monthly = daily_clean.resample("ME").last()
monthly.index.name = "Date"

assert len(monthly) == 228, f"Expected 228 months, got {len(monthly)}"
print(f"  Monthly observations: {len(monthly)}  (Apr 2006 – Mar 2025)")

monthly.to_csv("data/monthly_panel.csv")
print("  Saved  data/monthly_panel.csv")
print(f"\n  First 3:\n{monthly.head(3).to_string()}")
print(f"\n  Last 3:\n{monthly.tail(3).to_string()}")


# ── 5. SUMMARY STATISTICS ────────────────────────────────────────────────────

print("\n[3/5] Computing summary statistics ...")


def summary_row(s: pd.Series, label: str) -> dict:
    d          = s.dropna()
    jb_stat, jb_p = stats.normaltest(d)
    return {
        "Series"   : label,
        "N"        : len(d),
        "Mean"     : round(d.mean(), 4),
        "Std Dev"  : round(d.std(ddof=1), 4),
        "Min"      : round(d.min(), 4),
        "Max"      : round(d.max(), 4),
        "Skewness" : round(float(stats.skew(d)), 4),
        "Ex.Kurt"  : round(float(stats.kurtosis(d)), 4),
        "JB stat"  : round(jb_stat, 4),
        "JB p-val" : round(jb_p, 4),
    }


stat_rows = []
for col in monthly.columns:
    stat_rows.append(summary_row(monthly[col], col))
for col in monthly.columns:
    stat_rows.append(summary_row(monthly[col].diff().dropna(), f"D.{col}"))

stats_df = pd.DataFrame(stat_rows).set_index("Series")
stats_df.to_csv("data/summary_stats.csv")
print(stats_df.to_string())
print("\n  Saved  data/summary_stats.csv")


# ── 6. UNIT ROOT TESTS (ADF + Phillips-Perron, scipy only) ──────────────────
# ADF  H0: unit root present  → t-stat < critical value → reject → stationary
# PP   H0: unit root present  → Newey-West HAC corrected t-stat

print("\n[4/5] Running unit root tests ...")

# MacKinnon (1994) critical values — constant + trend specification
ADF_CV = {"1%": -3.96, "5%": -3.41, "10%": -3.13}


def _p_label(t: float) -> str:
    if   t < ADF_CV["1%"]:  return "< 0.01"
    elif t < ADF_CV["5%"]:  return "< 0.05"
    elif t < ADF_CV["10%"]: return "< 0.10"
    else:                    return "> 0.10"


def _build_adf_X(y: np.ndarray, lag: int) -> tuple:
    """
    Build OLS design matrix for ADF regression:
      Δy_t = α + βt + γy_{t-1} + Σ δ_i Δy_{t-i} + ε
    Returns (X, dy_t, gamma_col_index)
    """
    n  = len(y)
    dy = np.diff(y)

    if lag > 0:
        dy_t    = dy[lag:]
        ylag    = y[lag: n - 1]
        lag_mat = np.column_stack(
            [dy[lag - i - 1: n - i - 2] for i in range(lag)]
        )
        T     = len(dy_t)
        t_vec = np.arange(1, T + 1, dtype=float)
        X     = np.column_stack([np.ones(T), t_vec, ylag, lag_mat])
    else:
        dy_t  = dy
        ylag  = y[:-1]
        T     = len(dy_t)
        t_vec = np.arange(1, T + 1, dtype=float)
        X     = np.column_stack([np.ones(T), t_vec, ylag])

    return X, dy_t, 2   # column index 2 = γ (on y_{t-1})


def _ols(X: np.ndarray, y: np.ndarray):
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    resid         = y - X @ beta
    return beta, resid


def _select_lag_aic(y: np.ndarray, max_lag: int) -> int:
    best = (np.inf, 0)
    for lag in range(0, max_lag + 1):
        try:
            X, dy_t, _ = _build_adf_X(y, lag)
            beta, resid = _ols(X, dy_t)
            T    = len(resid)
            k    = X.shape[1]
            aic  = np.log(resid @ resid / T) + 2 * k / T
            if aic < best[0]:
                best = (aic, lag)
        except Exception:
            pass
    return best[1]


def adf_test(series: pd.Series, label: str) -> dict:
    y       = series.dropna().values.astype(float)
    max_lag = max(1, int(12 * (len(y) / 100) ** 0.25))
    lag     = _select_lag_aic(y, max_lag)

    X, dy_t, g_idx = _build_adf_X(y, lag)
    beta, resid    = _ols(X, dy_t)

    T      = len(resid)
    k      = X.shape[1]
    s2     = resid @ resid / (T - k)
    XtXinv = np.linalg.inv(X.T @ X)
    se     = np.sqrt(s2 * XtXinv[g_idx, g_idx])
def pp_test(series: pd.Series, label: str) -> dict:
    """
    Phillips-Perron Z_t statistic using Newey-West long-run variance.
    """
    y = series.dropna().values.astype(float)

    # Lag-0 ADF regression (no augmentation)
    X, dy_t, g_idx = _build_adf_X(y, 0)
        "CV 5%"     : ADF_CV["5%"],
        "Conclusion": "Stationary" if t_stat < ADF_CV["5%"] else "Unit root",
    }


def pp_test(series: pd.Series, label: str) -> dict:
    """
    Phillips-Perron Z_t statistic using Newey-West long-run variance.
    """
    y = series.dropna().values.astype(float)
    n = len(y)

    # Lag-0 ADF regression (no augmentation)
    X, dy_t, g_idx = _build_adf_X(y, 0)
    beta, resid    = _ols(X, dy_t)

    T  = len(resid)
    k  = X.shape[1]
    s2 = resid @ resid / (T - k)

    # Newey-West bandwidth (Andrews 1991)
    bw = max(1, int(4 * (T / 100) ** (2 / 9)))

    # Long-run variance via Bartlett kernel
    lrv = resid @ resid / T
    for j in range(1, bw + 1):
        w   = 1.0 - j / (bw + 1)
        cov = resid[j:] @ resid[: T - j] / T
        lrv += 2 * w * cov

    XtXinv = np.linalg.inv(X.T @ X)
    se_ols = np.sqrt(s2  * XtXinv[g_idx, g_idx])
    se_pp  = np.sqrt(lrv * XtXinv[g_idx, g_idx])

    gamma  = beta[g_idx]
    t_ols  = gamma / se_ols
    # Z_t = (s / sqrt(lrv)) * t_ols - correction term
    lrv_correction = (lrv - s2) / (2 * np.sqrt(lrv / (XtXinv[g_idx, g_idx])))
    t_pp = np.sqrt(s2 / lrv) * t_ols - lrv_correction * 0   # simplified Z_t
    t_pp = gamma / se_pp   # direct HAC-corrected t-statistic

    return {
        "Series"    : label,
        "Test"      : "PP",
        "Statistic" : round(t_pp, 4),
        "p-value"   : _p_label(t_pp),
        "Lags"      : bw,
        "CV 1%"     : ADF_CV["1%"],
        "CV 5%"     : ADF_CV["5%"],
        "Conclusion": "Stationary" if t_pp < ADF_CV["5%"] else "Unit root",
    }


ur_rows = []
for col in monthly.columns:
    ur_rows.append(adf_test(monthly[col],                    col))
    ur_rows.append(pp_test (monthly[col],                    col))
    ur_rows.append(adf_test(monthly[col].diff().dropna(),  f"D.{col}"))
    ur_rows.append(pp_test (monthly[col].diff().dropna(),  f"D.{col}"))

ur_df = pd.DataFrame(ur_rows)
ur_df.to_csv("data/stationarity_report.csv", index=False)
print(ur_df[["Series","Test","Statistic","p-value","Conclusion"]].to_string(index=False))
print("\n  Saved  data/stationarity_report.csv")


# ── 7. PLOTS ─────────────────────────────────────────────────────────────────

print("\n[5/5] Generating plots ...")

plt.rcParams.update({
    "font.family"       : "DejaVu Sans",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : False,
})

ordered_cols = [c for c in ["Y1","Y3","Y5","Y10","Repo"] if c in monthly.columns]

# ── 7a. Yield levels ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

for col in ordered_cols:
    ax.plot(monthly.index, monthly[col],
            label=MATURITY_LABELS.get(col, col),
            color=COLORS.get(col, "gray"),
            linestyle="--" if col == "Repo" else "-",
            linewidth=1.6 if col == "Repo" else 2.0)

# Mark key RBI policy episodes
episodes = {
    "GFC\n2008"       : "2008-10-01",
    "Rajan\n2013"     : "2013-09-04",
    "COVID cut\n2020" : "2020-03-27",
    "Tightening\n2022": "2022-05-04",
}
ylo, yhi = ax.get_ylim()
for ep, ds in episodes.items():
    xv = pd.to_datetime(ds)
    if monthly.index.min() <= xv <= monthly.index.max():
        ax.axvline(xv, color="#aaaaaa", linewidth=0.8, linestyle=":")
        ax.text(xv, yhi * 0.98, ep, fontsize=7.5, color="#666666",
                ha="center", va="top", linespacing=1.3)

ax.set_title("Indian G-Sec Yields and RBI Repo Rate — Monthly (Apr 2006 – Mar 2025)",
             fontsize=12, fontweight="bold", pad=12)
ax.set_ylabel("Yield / Rate (%)", fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.legend(loc="upper right", fontsize=9, framealpha=0.6, edgecolor="#cccccc")
ax.grid(axis="y", alpha=0.25, linewidth=0.5)
plt.tight_layout()
plt.savefig("plots/yield_series.png", dpi=180, bbox_inches="tight")
plt.close()
print("  plots/yield_series.png")

# ── 7b. Monthly changes ───────────────────────────────────────────────────────
n_p   = len(ordered_cols)
fig, axes = plt.subplots(n_p, 1, figsize=(13, 2.8 * n_p), sharex=True)
if n_p == 1:
    axes = [axes]

for ax, col in zip(axes, ordered_cols):
    d     = monthly[col].diff().dropna()
    color = COLORS.get(col, "steelblue")
    ax.bar(d.index, d.clip(lower=0).values, width=25, color=color,     alpha=0.75)
    ax.bar(d.index, d.clip(upper=0).values, width=25, color="#e74c3c", alpha=0.6)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_ylabel(f"Δ {MATURITY_LABELS.get(col, col)}\n(pp)", fontsize=8.5)
    ax.grid(axis="y", alpha=0.2, linewidth=0.5)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
fig.suptitle("Monthly Changes in Yields and Repo Rate (percentage points)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/yield_changes.png", dpi=180, bbox_inches="tight")
plt.close()
print("  plots/yield_changes.png")

# ── 7c. Correlation heatmap ───────────────────────────────────────────────────
col_labels = [MATURITY_LABELS.get(c, c) for c in ordered_cols]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (data, title) in zip(axes, [
    (monthly[ordered_cols],               "Levels"),
    (monthly[ordered_cols].diff().dropna(), "First Differences"),
]):
    corr = data.corr()
    sns.heatmap(corr, ax=ax, annot=True, fmt=".3f",
                cmap="RdYlGn", vmin=-1, vmax=1,
                linewidths=0.5, linecolor="white",
                xticklabels=col_labels, yticklabels=col_labels,
                annot_kws={"size": 9},
                cbar_kws={"shrink": 0.75})
    ax.set_title(f"Correlation — {title}", fontsize=11,
                 fontweight="bold", pad=10)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.tick_params(axis="y", rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig("plots/correlation_heatmap.png", dpi=180, bbox_inches="tight")
plt.close()
print("  plots/correlation_heatmap.png")


# ── 8. DONE ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  PRE-PROCESSING COMPLETE")
print("=" * 62)
print(f"  data/daily_panel.csv          {len(daily_clean):>5,} rows × {daily_clean.shape[1]} cols")
print(f"  data/monthly_panel.csv        {len(monthly):>5,} rows × {monthly.shape[1]} cols")
print(f"  data/summary_stats.csv        descriptive statistics")
print(f"  data/stationarity_report.csv  ADF + PP unit root tests")
print(f"  plots/yield_series.png        yield level chart")
print(f"  plots/yield_changes.png       monthly change bars")
print(f"  plots/correlation_heatmap.png")
print()
print("  NEXT  add 1Y/3Y/5Y paths to RAW_FILES dict at top of script")
print("  THEN  run:  python3 dns_model.py")
print("=" * 62)

SyntaxError: closing parenthesis '}' does not match opening parenthesis '(' on line 139 (3255177849.py, line 312)